In [1]:
from maite_datasets.object_detection import SeaDrone

# Downloads to ./data/seadrone on first run (~1.2 GB for the validation split).
seadrone = SeaDrone(root="./data", image_set="val", download=True)
print(f"{len(seadrone)} images")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1547 images


In [2]:
from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import DatasetProtocolConfig, SourceConfig, TaskConfig, ViewConfig, ViewOperation
from dataeval_flow.workflows.metadata_triage import MetadataTriageConfig

triage_workflow = MetadataTriageConfig(
    name="triage",
    max_examples=6,  # distinct values shown per column in the report
)

task = TaskConfig(
    name="triage_seadrone",
    workflow="triage",
    sources="seadrone_val",
)

config = PipelineConfig(
    datasets=[DatasetProtocolConfig(name="seadrone", dataset=seadrone)],
    views=[
        ViewConfig(
            name="sample",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 0}),
                ViewOperation(type="Limit", params={"size": 200}),
            ],
        )
    ],
    sources=[SourceConfig(name="seadrone_val", dataset="seadrone", view="sample")],
    workflows=[triage_workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config)
print(f"success={result.success}  health={result.health['status']}")

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_binning.py:420: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_binning.py:420: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `object_id`, `object_size`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flo

success=True  health=warning


In [4]:
print(result.report())


  METADATA TRIAGE
  Workflow:  triage (metadata-triage)
  Timestamp: 2026-10-01T07:19:28.757195+00:00
  Duration:  14.00s
  Source:    seadrone_val (seadrone[sample])

  Steps: 2 ran

  SUMMARY
  Unreadable factors ......................................... 3 factors  [!!]
  Columns dominated by one value ............................. 5 factors  [..]
  Degenerate factors ......................................... 1 factors  [..]
  Unpinned continuous bins .................................. 10 factors  [..]
  Unpinned categorical vocabularies .......................... 5 factors  [..]
  Suggested policy .............. add to configuration under `metadata:`  [..]
  Verified ................................................. 5 recovered  [..]
  Recommended policy ............ pins 17 factors as read from this data  [..]

  Health: 1 warning(s) [!!] — review flagged findings

  UNREADABLE FACTORS                                                   3 factors
  date_time — [blocking] cardinality

In [5]:
triage = result.steps["triage"].output.data()

In [6]:
latitude = next(f for f in triage["findings"] if f.factor == "latitude")

print("category :", latitude.category)
print("severity :", latitude.severity)
print("reasons  :", latitude.reasons)
print("suggested:", latitude.suggestion.corrections)
print("runnable :", latitude.suggestion.complete)

category : unreadable
severity : blocking
reasons  : ('mixed_types',)
suggested: [{'kind': 'remap', 'factor': 'latitude', 'rules': [{'match': 'N', 'to': None}]}]
runnable : False


In [7]:
print(triage["suggested_policy_yaml"])

metadata:
- name: standard
  corrections:
  - kind: parse_datetime
    factor: date_time
    every: day
  - kind: remap
    factor: latitude
    rules:
    - match: N
      to: null        # TODO
  - kind: remap
    factor: longitude
    rules:
    - match: E
      to: null        # TODO
  - kind: remap
    factor: altitude
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: compass_heading
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: gimbal_heading
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: gimbal_pitch
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: speed
    rules:
    - match: -1.0
      to: null        # TODO
  exclude:
  - object_id
  continuous_factor_bins:
    frame: 5
    object_size: 12
    xspeed: 2
    yspeed: 2



In [8]:
print(triage["recommended_policy_yaml"])

# Every cut and vocabulary below was read from this data. Pinning them keeps later runs comparable, and it also
# commits them to this data's assumptions. If this data does not represent the data you expect, results computed under
# this policy can be invalid or misleading. Review each factor before you commit it.
metadata:
- name: standard
  corrections:
  - kind: parse_datetime
    factor: date_time
    every: day
  - kind: remap
    factor: latitude
    rules:
    - match: N
      to: .nan  # dropped by default: decide what 'N' means
  - kind: remap
    factor: longitude
    rules:
    - match: E
      to: .nan  # dropped by default: decide what 'E' means
  exclude:
  - object_id
  continuous_factor_bins:
    frame: [-.inf, 2630.75, 5083.5, 7536.25, 9989.0, 12441.75, 14894.5, 17347.25, .inf]  # seen 178 to 19800; the outer bins are open, so values beyond this range join them
    object_size: [-.inf, 42619.625, 85129.25, 127638.875, 170148.5, 212658.125, 255167.75, 297677.375, 340187

In [9]:
from dataeval_flow.config import MetadataPolicyConfig

policy = MetadataPolicyConfig.model_validate(
    {
        "name": "seadrone",
        "corrections": [
            {"kind": "parse_datetime", "factor": "date_time", "every": "day"},
            # Map hemisphere letters to NaN (missing values)
            {"kind": "remap", "factor": "latitude", "rules": [{"match": "N", "to": float("nan")}]},
            {"kind": "remap", "factor": "longitude", "rules": [{"match": "E", "to": float("nan")}]},
            # Map -1.0 sentinel values to NaN
            *(
                {"kind": "remap", "factor": name, "rules": [{"match": -1.0, "to": float("nan")}]}
                for name in ("altitude", "compass_heading", "gimbal_heading", "gimbal_pitch", "speed")
            ),
        ],
        # Exclude object identifier
        "exclude": ["object_id"],
        "continuous_factor_bins": triage["suggested_policy"]["continuous_factor_bins"],
    }
)

corrected = config.model_copy(
    update={
        "metadata": [policy],
        "workflows": [triage_workflow.model_copy(update={"metadata": "seadrone"})],
    }
)
result2 = run_task(task, corrected)
print(f"success={result2.success}  health={result2.health['status']}")

success=True  health=ok


/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_binning.py:420: UserWarning: `altitude`, `compass_heading`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude` and `speed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_binning.py:420: UserWarning: Declared cuts left bins unused: frame (4 of 5 bins hold rows), object_size (9 of 12 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  factor_info = metadata.factor_info


In [10]:
after = result2.steps["triage"].output.data()
print(f"factors : {triage['factor_count']} -> {after['factor_count']}")
print(f"findings: {len(triage['findings'])} -> {len(after['findings'])}")
print(f"blocking: {triage['counts'].get('blocking', 0)} -> {after['counts'].get('blocking', 0)}")

factors : 15 -> 17
findings: 24 -> 21
blocking: 3 -> 0


In [11]:
json_str = result.export(fmt="json")
print(f"JSON output: {len(json_str)} characters")

JSON output: 342015 characters
